In [1]:
import findspark
findspark.init()

from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

conf = SparkConf().setAppName("550").setMaster("local[4]")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/16 06:23:14 WARN Utils: Your hostname, de24, resolves to a loopback address: 127.0.1.1; using 192.168.0.103 instead (on interface enp0s3)
25/08/16 06:23:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/16 06:23:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
'''
+--------------+---------+
| Column Name  | Type    |
+--------------+---------+
| player_id    | int     |
| device_id    | int     |
| event_date   | date    |
| games_played | int     |
+--------------+---------+
(player_id, event_date) is the primary key (combination of columns with unique values) of this table.
This table shows the activity of players of some games.
Each row is a record of a player who logged in and played a number of games (possibly 0) before logging out on someday 
using some device.
 

Write a solution to report the fraction of players that logged in again on the day after the day they first logged in, 
rounded to 2 decimal places. In other words, you need to count the number of players that logged in for at least two 
consecutive days starting from their first login date, then divide that number by the total number of players.

The result format is in the following example.

 

Example 1:

Input: 
Activity table:
+-----------+-----------+------------+--------------+
| player_id | device_id | event_date | games_played |
+-----------+-----------+------------+--------------+
| 1         | 2         | 2016-03-01 | 5            |
| 1         | 2         | 2016-03-02 | 6            |
| 2         | 3         | 2017-06-25 | 1            |
| 3         | 1         | 2016-03-02 | 0            |
| 3         | 4         | 2018-07-03 | 5            |
+-----------+-----------+------------+--------------+
Output: 
+-----------+
| fraction  |
+-----------+
| 0.33      |
+-----------+
Explanation: 
Only the player with id 1 logged back in after the first day he had logged in so the answer is 1/3 = 0.33
'''

In [16]:
data = [
(1,2,'2016-03-01',5),
(1,2,'2016-03-02',6),
(2,3,'2017-06-25',1),
(3,1,'2016-03-02',0),
(3,4,'2018-07-03',5)    
]
schema = ['player_id','device_id','event_date','games_played']

In [17]:
df = spark.createDataFrame(data = data, schema = schema)
df.show()

+---------+---------+----------+------------+
|player_id|device_id|event_date|games_played|
+---------+---------+----------+------------+
|        1|        2|2016-03-01|           5|
|        1|        2|2016-03-02|           6|
|        2|        3|2017-06-25|           1|
|        3|        1|2016-03-02|           0|
|        3|        4|2018-07-03|           5|
+---------+---------+----------+------------+



In [18]:
firstday_df = df.groupBy(F.col("player_id"))\
                .agg(F.min(F.col("event_date")).alias("first_day"))

distinct_count = df.select(F.count_distinct(F.col("player_id"))).collect()[0][0]

df.alias("t").join(firstday_df.alias("t1"),
                   F.col("t.player_id") == F.col("t1.player_id"),
                   'left'
                  )\
            .where(F.date_diff(F.col("t.event_date"),F.col("t1.first_day")) == 1)\
            .select(F.round(
                    F.count("*")/distinct_count
                    ,2).alias("fraction"))\
            .show()
            

+--------+
|fraction|
+--------+
|    0.33|
+--------+



## SQL Solution

### BEST SOLUTION
<pre>
WITH first_day_data
AS (
	SELECT player_id
		,min(event_date) AS first_day
	FROM Activity
	GROUP BY player_id
	)
SELECT round(count(*) / (
			SELECT count(DISTINCT player_id)
			FROM Activity
			), 2) AS fraction
FROM Activity t1
INNER JOIN first_day_data t2 ON DATEDIFF(t1.event_date, t2.first_day) = 1
	AND t1.player_id = t2.player_id;
</pre>


### ALTERNATE SOLUTION
<pre>
WITH LAGGED as (
SELECT player_id, device_id, event_date, games_played,
       LAG(event_date,1,NULL) OVER(PARTITION BY player_id ORDER BY event_date asc) as LAG_1
FROM medium_550),
REGULAR_PALYERS as (    
SELECT player_id, SUM(CASE WHEN DATEDIFF(event_date,LAG_1) = 1 THEN 1 ELSE 0 END) as consugutive
FROM LAGGED 
GROUP BY player_id)
SELECT ROUND(COUNT(player_id)/(SELECT  COUNT(DISTINCT player_id) FROM medium_550),2) as fraction
FROM REGULAR_PALYERS
WHERE consugutive >= 1;
</pre>